In [1]:
import random
import numpy as np

# --- 1. USER INPUT PIPELINE ---
print("="*50)
print("  ADVANCED MULTI-OBJECTIVE FITNESS ROUTINE GENERATOR")
print("="*50)
try:
    user_age = float(input("Enter user age (e.g., 38): "))
    print("\nSelect Primary Metabolic Context:")
    print("1: Aggressive Fat Loss / Metabolic Reset")
    print("2: Hypertrophy / Lean Muscle Accrual")
    print("3: Sustainable Longevity & Vitality")
    goal_choice = int(input("Enter choice (1-3): "))

    max_days = int(input("\nMaximum workout availability per week? (1-7 days): "))
    user_max_days = max(1, min(7, max_days))
    required_rest_days = 7 - user_max_days
except ValueError:
    print("\n[!] Invalid input detected. Defaulting to: Age 35, Longevity, Max 5 Workout Days.")
    user_age = 35.0
    goal_choice = 3
    required_rest_days = 2

# Biological scaling factor (accounts for natural post-30 recovery windows)
age_scalar = max(1.0, user_age / 30.0)

# --- 2. MULTI-OBJECTIVE FITNESS EVALUATION ENGINE ---
def evaluate_objectives(chromosome):
    """
    Evaluates a fitness chromosome across two competing biological axes.
    Chromosome: [Cardio_hrs, Strength_hrs, Mobility_hrs, Calorie_delta, Rest_days]

    Returns:
        tuple: (physiological_adaptation_score, systemic_stress_score)
        Goal -> MAXIMIZE Adaptation, MINIMIZE Stress
    """
    cardio, strength, mobility, calories, rest = chromosome

    # ---------------------------------------------------------
    # OBJECTIVE 1: MAXIMIZE PHYSIOLOGICAL ADAPTATION (Higher is Better)
    # ---------------------------------------------------------
    adaptation = 0.0

    # Resistance training benefit curve (Post-30 crucial pivot for osteopenia/sarcopenia prevention)
    ideal_strength_floor = 2.5 * age_scalar
    if strength >= ideal_strength_floor:
        adaptation += 40.0 + (strength - ideal_strength_floor) * 2
    else:
        adaptation += (strength / ideal_strength_floor) * 40.0

    # Cardiovascular/Aerobic capacity adaptation
    if goal_choice == 1: # Fat loss priority
        adaptation += min(cardio, 4.0) * 10
    else:
        adaptation += min(cardio, 2.5) * 12

    # Joint integrity and functional range of motion (Mobility)
    adaptation += min(mobility, 2.0) * 15

    # ---------------------------------------------------------
    # OBJECTIVE 2: MINIMIZE SYSTEMIC STRESS & CORTISOL (Lower is Better)
    # ---------------------------------------------------------
    stress = 10.0 # Base biological stress floor

    total_training_hours = cardio + strength + mobility
    actual_workout_days = 7 - rest

    # Volume density stress (Overtraining penalties scaled by age)
    if actual_workout_days > 0:
        daily_average_volume = total_training_hours / actual_workout_days
        if daily_average_volume > 2.0:
            stress += (daily_average_volume - 2.0) * 25 * age_scalar

    # Overtraining without operational windows
    if rest < required_rest_days:
        stress += (required_rest_days - rest) * 35
    if rest < 1:
        stress += 60 # Severe biological strain penalty if 0 rest days are selected

    # Cardiovascular cortisol exhaustion curve
    if cardio > 4.5:
        stress += (cardio - 4.5) * 15

    # Metabolic / Caloric strain calculations
    if goal_choice == 1: # Fat Loss
        if calories < -500: # Crash dieting penalty
            stress += abs(calories + 500) * 0.15
        elif calories > 0: # Off-target penalty
            stress += calories * 0.1
    elif goal_choice == 2: # Muscle Gain
        if calories < -100: # Catabolic risk
            stress += abs(calories + 100) * 0.2
    else: # Longevity
        if abs(calories) > 200:
            stress += (abs(calories) - 200) * 0.15

    return round(adaptation, 2), round(max(stress, 0.0), 2)


# --- 3. GENETIC OPERATORS & SELECTION ENGINE ---

def create_individual():
    return [
        random.uniform(0.0, 6.0),                  # Cardio hours
        random.uniform(0.0, 6.0),                  # Strength hours
        random.uniform(0.5, 4.0),                  # Mobility hours
        random.uniform(-600, 400),                 # Calorie balance delta
        random.randint(required_rest_days, 4)      # Rest days per week
    ]

def crossover(p1, p2):
    idx = random.randint(1, 4)
    return p1[:idx] + p2[idx:], p2[:idx] + p1[idx:]

def mutate(ind, mutation_rate=0.25):
    if random.random() < mutation_rate:
        gene = random.randint(0, 4)
        if gene == 0: ind[0] = max(0.0, ind[0] + random.uniform(-0.7, 0.7))
        elif gene == 1: ind[1] = max(0.0, ind[1] + random.uniform(-0.7, 0.7))
        elif gene == 2: ind[2] = max(0.0, ind[2] + random.uniform(-0.4, 0.4))
        elif gene == 3: ind[3] = ind[3] + random.uniform(-75, 75)
        elif gene == 4: ind[4] = max(required_rest_days, min(4, ind[4] + random.choice([-1, 1])))
    return ind

def pareto_tournament_selection(population, scores, pool_size=3):
    """Selects high-performing individuals using multi-objective dominance."""
    pool_indices = random.sample(range(len(population)), pool_size)

    # Sort selected pool by checking Pareto dominance attributes
    # Best = Highest Adaptation AND Lowest Stress
    best_idx = pool_indices[0]
    for idx in pool_indices[1:]:
        ad_best, st_best = scores[best_idx]
        ad_curr, st_curr = scores[idx]

        # If the current candidate dominates the current best, swap them
        if (ad_curr >= ad_best and st_curr <= st_best) and (ad_curr > ad_best or st_curr < st_best):
            best_idx = idx

    return population[best_idx]


# --- 4. OPTIMIZATION RUNTIME EXECUTION ---

def run_evolutionary_optimizer(pop_size=80, generations=120):
    population = [create_individual() for _ in range(pop_size)]

    print(f"\n[+] Executing GA optimization loop across {generations} generations...")

    for gen in range(generations):
        # 1. Compute biological objective matrices
        scores = [evaluate_objectives(ind) for ind in population]

        # 2. Re-populate next tracking generation
        new_population = []

        # Elitism extraction step for Pareto front integrity
        sorted_indices = sorted(range(len(scores)), key=lambda i: (scores[i][0], -scores[i][1]), reverse=True)
        new_population.append(population[sorted_indices[0]]) # Best Adaptation Individual
        new_population.append(population[sorted_indices[1]]) # High balance safety profile individual

        while len(new_population) < pop_size:
            parent1 = pareto_tournament_selection(population, scores)
            parent2 = pareto_tournament_selection(population, scores)

            c1, c2 = crossover(parent1, parent2)
            new_population.append(mutate(c1))
            if len(new_population) < pop_size:
                new_population.append(mutate(c2))

        population = new_population

    # Post-processing generation array: Filter out unique optimized balance variants
    final_scores = [evaluate_objectives(ind) for ind in population]

    print("\n" + "="*55)
    print("         GENETIC GENERATION SYSTEM RUN COMPLETE")
    print("="*55)

    # Capture highly distinct profiles along our optimized Pareto front
    seen_profiles = set()
    profile_count = 1

    for idx in sorted_indices[:15]: # Review top elite variations
        ind = population[idx]
        cardio, strength, mobility, calories, rest = ind
        # Group similar routines together using rounded values to ensure a diverse output menu
        profile_key = (round(cardio, 1), round(strength, 1), int(rest))

        if profile_key not in seen_profiles and profile_count <= 3:
            seen_profiles.add(profile_key)
            adapt, stress = final_scores[idx]

            print(f"\n👉 OPTIMIZED ROUTINE OPTION #{profile_count} (Adaptation Score: {adapt} | Stress Score: {stress})")
            print(f"  • Cardio Training Volume:   {cardio:.1f} hours / week")
            print(f"  • Resistance/Strength Work: {strength:.1f} hours / week")
            print(f"  • Mobility & Recovery:      {mobility:.1f} hours / week")
            print(f"  • Daily Caloric Adjustment: {calories:+.0f} kcal / day")
            print(f"  • Target Rest Windows:      {int(rest)} days / week ({7 - int(rest)} active days)")
            profile_count += 1

    print("="*55)

if __name__ == "__main__":
    run_evolutionary_optimizer()


  ADVANCED MULTI-OBJECTIVE FITNESS ROUTINE GENERATOR
Enter user age (e.g., 38): 32

Select Primary Metabolic Context:
1: Aggressive Fat Loss / Metabolic Reset
2: Hypertrophy / Lean Muscle Accrual
3: Sustainable Longevity & Vitality
Enter choice (1-3): 1

Maximum workout availability per week? (1-7 days): 5

[+] Executing GA optimization loop across 120 generations...

         GENETIC GENERATION SYSTEM RUN COMPLETE

👉 OPTIMIZED ROUTINE OPTION #1 (Adaptation Score: 131.92 | Stress Score: 61.37)
  • Cardio Training Volume:   4.0 hours / week
  • Resistance/Strength Work: 13.6 hours / week
  • Mobility & Recovery:      2.0 hours / week
  • Daily Caloric Adjustment: -116 kcal / day
  • Target Rest Windows:      2 days / week (5 active days)

👉 OPTIMIZED ROUTINE OPTION #2 (Adaptation Score: 130.9 | Stress Score: 60.81)
  • Cardio Training Volume:   3.9 hours / week
  • Resistance/Strength Work: 13.6 hours / week
  • Mobility & Recovery:      2.0 hours / week
  • Daily Caloric Adjustment: -1